<a href="https://colab.research.google.com/github/DrGPCR/NEUR201/blob/main/Unit_3/notebooks/Unit_3_Statistics_Assignment_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 3 Statistics Assignment — Is the New Compound's Effect Real?

**NEUR 201 — Research Methods & Data Analysis for Cellular Neuroscience**  ·  *Assignment (10 points)*

**Name:** *(double-click to type)*  **Date:** *(double-click to type)*

---

## Background

The Data Analysis assignment gave you **one** slice per condition. That is a measurement, not an
experiment — image a different slice and the number would come out differently.

So the new compound was tested on more tissue: **12 CONTROL slices** (aCSF vehicle) and **12 DRUG
slices** (the new compound), each imaged for six minutes. Each slice was reduced to one number, its
**mean ΔF/F₀ (%) during the application period**: 0% is no change from that slice's own baseline, a
positive value means it became more active, and a negative value means it became less active.

The lab's question:

> **Does the new compound change calcium activity — or could the gap between the groups be chance?**

Each **slice** is one independent data point, so **n is the number of slices** (12 per group), not the
number of cells. Treating cells as independent would be **pseudoreplication**.

The two groups are **independent** (a slice is either drug or vehicle, never both), so the right test is
an **independent-samples t-test**.

**Symbols:** $\bar{X}$ = sample mean · $s$ = sample SD · $n$ = number of slices · $SE$ = standard
error of the mean · $t$ = t-score · $df$ = degrees of freedom · $H_0$, $H_a$ = null and alternative
hypotheses · $p$ = p-value · $\alpha$ = alpha level

## Hypothesis testing in a nutshell


Every t-test chooses between two explanations for the gap between two sample means:

| | Claim | Result if true |
|---|---|---|
| **Null hypothesis, $H_0$** | The gap is just **random sampling error** (chance). | statistically **insignificant** |
| **Alternative hypothesis, $H_a$** | The gap **cannot** be explained by chance alone. | statistically **significant** |

A **two-tailed** $H_a$ says the means are *different* without predicting which is larger. For two groups
with population means $\mu_1$ and $\mu_2$ it takes this form:

$$ H_0: \mu_1 = \mu_2 \qquad\qquad H_a: \mu_1 \neq \mu_2 $$

A **one-tailed** $H_a$ predicts a direction (larger *or* smaller) and must be chosen **before** seeing the
data. We use **two-tailed** tests throughout.

**The decision rule.** The cut-off p-value is the **alpha level ($\alpha$)**, conventionally **0.05**:

| If | Decision | Wording |
|---|---|---|
| $p < \alpha$ | **reject $H_0$** | the difference is **statistically significant** |
| $p \geq \alpha$ | **fail to reject $H_0$** | chance cannot be ruled out: **not** significant |


## How this assignment works

**Run every cell in order, from the top** (click a cell, press **Shift + Enter**).

| | What to do |
|---|---|
| **Steps 1–5** | Run the cells so the outputs show in your PDF. Follow the procedure from class: SE of each sample → t-score → df → p-value → graph. |
| **⚙️ Try it** | One, in Step 5: change `alpha` and re-run. Discussion question 4 depends on it. |
| **💬 Discussion** | At the very end. Give numbers exactly as the notebook prints them; the other answers are short sentences. |

**Points (10 total):** each of Steps 1–5 is 1 point, earned by running its cells so the outputs show in
your PDF; the Discussion is worth 5 points (marked on each question).

**To submit:** answer the Discussion, click **Runtime → Run all**, then **File → Print → Save as PDF**
and upload the PDF to Canvas.

---

## Step 1 — Load the data [1 pt]

Load the twelve slices per group and print each group's mean.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA = "https://raw.githubusercontent.com/DrGPCR/NEUR201/main/Unit_3/data/"

try:
    data = pd.read_csv(DATA + "dFF_per_slice_assignment.csv")
except Exception:
    # Safety net: an exact copy of the same file, used only if the download fails.
    _CON  = [0.03, 0.38, 0.24, -1.13, -1.52, 0.45, 0.56, 2.48, -0.79, 0.53, 0.23, -0.28]
    _DRUG = [-6.55, -7.12, 3.16, -4.16, 3.39, -0.56, -4.55, -1.32, -7.20, -4.08, -2.60, -1.68]
    data = pd.DataFrame({"Group": ["CONTROL"] * 12 + ["DRUG"] * 12,
                         "Slice": list(range(1, 13)) * 2,
                         "dFF_percent": _CON + _DRUG})

control = data.loc[data["Group"] == "CONTROL", "dFF_percent"].to_numpy()
drug    = data.loc[data["Group"] == "DRUG",    "dFF_percent"].to_numpy()

print("CONTROL slices:", len(control), "  DRUG slices:", len(drug))
print("CONTROL mean: %5.2f %%" % control.mean())
print("DRUG mean:    %5.2f %%" % drug.mean())
print("Difference (DRUG - CONTROL): %.2f percentage points" % (drug.mean() - control.mean()))

## Step 2 — Standard error of each sample [1 pt]

**Step 1 of an independent-samples t-test: find the standard error of each sample separately.**

$$ SE = \frac{s}{\sqrt{n}} $$

where $s$ is the sample standard deviation and $n$ is the number of **slices** in that group.

The **SD** ($s$) describes how spread out the **slices** are; the **SE** describes how uncertain
each group's **mean** is.

In [ ]:
n_control = len(control)
n_drug    = len(drug)

s_control = control.std(ddof=1)     # sample standard deviation (divides by n - 1)
s_drug    = drug.std(ddof=1)

se_control = s_control / np.sqrt(n_control)     # SE = s / sqrt(n)
se_drug    = s_drug    / np.sqrt(n_drug)

print("CONTROL:  s = %.2f,  n = %d   ->   SE = %.2f / sqrt(%d) = %.2f" % (s_control, n_control, s_control, n_control, se_control))
print("DRUG:     s = %.2f,  n = %d   ->   SE = %.2f / sqrt(%d) = %.2f" % (s_drug, n_drug, s_drug, n_drug, se_drug))

## Step 3 — Combined standard error and the t-score [1 pt]

**Step 2: combine the two standard errors.** Both group means wobble, so the standard error of the
**difference** between them is

$$ SE_{\bar{X}_1 - \bar{X}_2} = \sqrt{SE_{\bar{X}_1}^2 + SE_{\bar{X}_2}^2} $$

**Step 3: the t-score.** Count the difference between the means in units of that standard error:

$$ t = \frac{\bar{X}_{\text{DRUG}} - \bar{X}_{\text{CONTROL}}}{SE_{\bar{X}_1 - \bar{X}_2}} $$

Read $t$ as a question: *how many standard errors apart did the two means land?* It is the same idea as
the one-sample t-score, with the difference between **two sample means** on top.

In [ ]:
# Combined standard error: both means wobble, so the squared errors add
se_difference = np.sqrt(se_control**2 + se_drug**2)

# The t-score: the difference between the means, counted in standard errors
difference = drug.mean() - control.mean()
t_score    = difference / se_difference

print("SE of the difference = sqrt(%.2f^2 + %.2f^2) = %.2f" % (se_control, se_drug, se_difference))
print()
print("Difference in means  = %.2f - (%.2f) = %.2f percentage points" % (drug.mean(), control.mean(), difference))
print("t = %.2f / %.2f = %.2f" % (difference, se_difference, t_score))
print()
print("The DRUG mean sits %.2f standard errors %s the CONTROL mean." % (abs(t_score), "above" if t_score > 0 else "below"))

## Step 4 — Degrees of freedom and the p-value [1 pt]

**Step 4: degrees of freedom.** We estimated **two** means, so we lose two degrees of freedom:

$$ df = n_1 + n_2 - 2 $$

**Step 5: the p-value.** The p-value is the probability of getting a t **at least this far from 0, in either
direction, if $H_0$ were true**, i.e. the area in **both tails** beyond $\pm t$. (You can always take the
**absolute value** of $t$ when finding p: the two tails are mirror images.)

Read that sentence twice, because almost everyone gets it backwards: p is the probability of **the data,
given $H_0$**, never the probability that $H_0$ is true.

Python's `scipy.stats` looks up the exact p-value for us. In class you used the printed t-table, which
gives a **range**; both answer the same question. The **critical t** is the t you would need to reach to
have $p = \alpha$ exactly. (We set $\alpha = 0.05$ here.)

In [ ]:
df_ours = n_control + n_drug - 2

p_value = 2 * stats.t.sf(abs(t_score), df_ours)     # area in BOTH tails beyond our t (two-tailed)

alpha  = 0.05                                       # the cut-off, set before the test
t_crit = stats.t.ppf(1 - alpha / 2, df_ours)        # the t you would need to reach p = alpha

# Check against scipy's built-in independent-samples t-test
t_scipy, p_scipy = stats.ttest_ind(drug, control)

print("df = %d + %d - 2 = %d" % (n_control, n_drug, df_ours))
print("t  = %.2f      (|t| = %.2f)" % (t_score, abs(t_score)))
print("p-value (two-tailed) = %.4f" % p_value)
print()
print("alpha = %g   ->   critical t = +/-%.2f" % (alpha, t_crit))
print()
print("Check with scipy:  t = %.2f,  p = %.4f" % (t_scipy, p_scipy))

## Step 5 — Present the result on the t-distribution [1 pt]

The curve shows how t would behave **if $H_0$ were true**. The **red** area (both tails beyond $\pm t$) is the **p-value**; the **grey** area is the rejection region for $\alpha$ (the part of the curve beyond the critical t).

> **⚙️ Try it — change alpha.** Run at `alpha = 0.05`, then `0.01`, then `0.001`, and note where your t lands relative to the grey region each time. Then set it back to **0.05**.

In [ ]:
alpha = 0.05        # ⚙️ Try it — 0.05, then 0.01, then 0.001. Set back to 0.05.

t_crit = stats.t.ppf(1 - alpha / 2, df_ours)        # recompute the critical t for this alpha

lim = max(5, abs(t_score) + 1)
x = np.linspace(-lim, lim, 800)
y = stats.t.pdf(x, df_ours)

plt.figure(figsize=(8, 4.2))
plt.plot(x, y, color="black")
plt.fill_between(x, y, where=np.abs(x) >= t_crit, color="lightgray",
                 label="rejection region (alpha = %g)" % alpha)
plt.fill_between(x, y, where=np.abs(x) >= abs(t_score), color="tomato", alpha=0.85,
                 label="p-value = %.4f (both tails)" % p_value)
plt.axvline(t_score,  color="darkred", lw=2, label="our t = %.2f" % t_score)
plt.axvline(-t_score, color="darkred", lw=1, ls=":")          # the mirror-image tail
for c in (-t_crit, t_crit):
    plt.axvline(c, color="gray", ls="--", lw=1)
plt.xlabel("t")
plt.ylabel("Probability density")
plt.title("t-distribution if H0 were true (df = %d)" % df_ours)
plt.legend(fontsize=8, loc="upper left")
plt.show()

print("t = %.2f,  df = %d,  p = %.4f,  alpha = %g" % (t_score, df_ours, p_value, alpha))

---

### 💬 Discussion

Use the values the notebook printed (do not round differently). Points are marked on each question
(5 in total).

1. **Hypotheses and t-score (Steps 3–4) [2 pts].** State the **two-tailed** null and alternative
   hypotheses for this experiment, in words or in symbols using $\mu_{\text{DRUG}}$ and
   $\mu_{\text{CONTROL}}$ *(1 pt)*. Then report the **t-score** comparing DRUG with CONTROL and the
   **degrees of freedom** *(1 pt)*.

2. **p-value (Step 4) [1 pt].** What is the p-value, to 4 decimal places?

3. **Decision at $\alpha = 0.05$ (Step 5) [1 pt].** Compare your p-value with $\alpha = 0.05$. Is the
   difference statistically significant? Say whether you **reject** or **fail to reject** $H_0$, and why.

4. **Try two different alphas (Step 5) [1 pt].** Use the Try it to set `alpha` to **0.01** and then
   **0.001**, re-running each time. For each one, do you **reject** or **fail to reject** $H_0$? Then
   explain, in one or two sentences, what your results mean **biologically** for the new compound's
   effect on calcium activity.

*(Before you submit: set `alpha = 0.05` in Step 5, then run all.)*

**Your answer:** *(double-click here to type)*

>
>
>
>
>

---
### Nice work — you're done!

Click **Runtime → Run all**, then **File → Print → Save as PDF**, and upload the PDF to Canvas.

This cell is empty on purpose. Do not delete.